In [1]:
from pathlib import Path
import json
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)

EXPERIMENTS_DIR = (
    PROJECT_ROOT
    / "experiments"
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPERIMENTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Inspectra — Baseline Training")
print("=" * 60)
print(
    "PyTorch:",
    torch.__version__
)
print(
    "CUDA:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "Device:",
    DEVICE
)

Inspectra — Baseline Training
PyTorch: 2.5.1+cu121
CUDA: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Device: 0


In [2]:
DATASET_TASKS = {
    "bottle": "detection",
    "pcb": "detection",
    "textile": "detection",
    "welding": "detection",
    "road": "classification",
    "steel": "segmentation",
}


def count_images(path):

    extensions = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".tif",
        ".tiff",
        ".webp",
    }

    if not path.exists():
        return 0

    return sum(
        1
        for file in path.rglob("*")
        if (
            file.is_file()
            and file.suffix.lower()
            in extensions
        )
    )


readiness = []

for dataset, task in DATASET_TASKS.items():

    root = (
        PROCESSED_DATA
        / dataset
    )

    train_count = count_images(
        root / "train"
    )

    val_count = count_images(
        root / "val"
    )

    test_count = count_images(
        root / "test"
    )

    ready = (
        train_count > 0
        and val_count > 0
        and test_count > 0
    )

    readiness.append(
        {
            "dataset": dataset,
            "task": task,
            "train": train_count,
            "val": val_count,
            "test": test_count,
            "ready": ready,
        }
    )


readiness_df = pd.DataFrame(
    readiness
)

display(
    readiness_df
)

,dataset,task,train,val,test,ready
0,bottle,detection,5420,1799,920,True
1,pcb,detection,8534,1066,1068,True
2,textile,detection,0,0,0,False
3,welding,detection,0,0,0,False
4,road,classification,28000,6000,6000,True
5,steel,segmentation,0,0,0,False


In [3]:
DETECTION_DATASETS = {
    "bottle": {
        "classes": [
            "Cap",
            "Missing",
            "Wrong bottle",
            "box",
        ],
        "model": "yolov8n.pt",
        "epochs": 50,
        "batch": 4,
        "imgsz": 640,
    },

    "pcb": {
        "classes": [
            "mouse_bite",
            "spur",
            "missing_hole",
            "short",
            "open_circuit",
            "spurious_copper",
        ],
        "model": "yolov8n.pt",
        "epochs": 50,
        "batch": 4,
        "imgsz": 640,
    },

    "textile": {
        "classes": [],
        "model": "yolov8n.pt",
        "epochs": 50,
        "batch": 4,
        "imgsz": 640,
    },

    "welding": {
        "classes": [
            "adj",
            "int",
            "geo",
            "pro",
            "non",
        ],
        "model": "yolov8n.pt",
        "epochs": 50,
        "batch": 4,
        "imgsz": 640,
    },
}

In [4]:
import yaml


def create_yolo_yaml(
    dataset_name,
    classes
):

    root = (
        PROCESSED_DATA
        / dataset_name
    )

    config = {
        "path": str(root.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": len(classes),
        "names": classes,
    }

    yaml_path = (
        root
        / "data.yaml"
    )

    with open(
        yaml_path,
        "w",
        encoding="utf-8"
    ) as file:

        yaml.safe_dump(
            config,
            file,
            sort_keys=False
        )

    return yaml_path

In [5]:
for dataset, config in DETECTION_DATASETS.items():

    if not config["classes"]:
        continue

    if not (
        PROCESSED_DATA
        / dataset
        / "train"
        / "images"
    ).exists():
        continue

    yaml_path = create_yolo_yaml(
        dataset,
        config["classes"]
    )

    print(
        f"{dataset:10} -> {yaml_path}"
    )

bottle     -> d:\Inspectra\datasets\processed\bottle\data.yaml
pcb        -> d:\Inspectra\datasets\processed\pcb\data.yaml
welding    -> d:\Inspectra\datasets\processed\welding\data.yaml


In [6]:
from ultralytics import YOLO


def train_detection_baseline(
    dataset_name,
    config
):

    dataset_root = (
        PROCESSED_DATA
        / dataset_name
    )

    train_dir = (
        dataset_root
        / "train"
        / "images"
    )

    val_dir = (
        dataset_root
        / "val"
        / "images"
    )

    if not train_dir.exists():
        print(
            f"{dataset_name}: training data missing"
        )
        return None

    if count_images(train_dir) == 0:
        print(
            f"{dataset_name}: no training images"
        )
        return None

    if count_images(val_dir) == 0:
        print(
            f"{dataset_name}: no validation images"
        )
        return None

    yaml_path = create_yolo_yaml(
        dataset_name,
        config["classes"]
    )

    output_dir = (
        EXPERIMENTS_DIR
        / dataset_name
        / "baseline"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    model = YOLO(
        config["model"]
    )

    start_time = time.time()

    results = model.train(
        data=str(yaml_path),
        epochs=config["epochs"],
        batch=config["batch"],
        imgsz=config["imgsz"],
        device=DEVICE,
        project=str(
            output_dir
        ),
        name="baseline",
        pretrained=True,
        workers=4,
        seed=42,
        deterministic=True,
        plots=True,
        save=True,
        verbose=True,
    )

    elapsed = (
        time.time()
        - start_time
    )

    print(
        f"\n{dataset_name.upper()}"
    )

    print(
        f"Training time: "
        f"{elapsed / 60:.2f} minutes"
    )

    print(
        "Best model:",
        output_dir
        / "baseline"
        / "weights"
        / "best.pt"
    )

    return results

In [7]:
bottle_result = train_detection_baseline(
    "bottle",
    DETECTION_DATASETS["bottle"]
)

Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\Inspectra\datasets\processed\bottle\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_sca

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

AMP: checks passed 
WARNING train: Slow image access detected (ping: 0.20.1 ms, read: 1.80.2 MB/s, size: 34.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
train: Scanning D:\Inspectra\datasets\processed\bottle\train\labels... 5420 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5420/5420 402.5it/s 13.5s0.1s
train: New cache created: D:\Inspectra\datasets\processed\bottle\train\labels.cache
WARNING val: Slow image access detected (ping: 0.20.1 ms, read: 3.60.7 MB/s, size: 41.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\Inspectra\datasets\processed\bottle\val\labels... 1799 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1799/1799 396.0it/s 4.5s0.1s
val: New cache created: D:\Inspectra\datasets\processed\bottle\val\labels.cache
optimizer: 'optimizer=auto' found, ignoring

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50     0.592G     0.7514      1.832      1.108         18        640: 100% ━━━━━━━━━━━━ 1355/1355 13.8it/s 1:380.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 15.7it/s 14.3s0.1s
                   all       1799       3479      0.821      0.775      0.814      0.661

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50     0.688G     0.6246      1.469      1.165         10        640: 0% ──────────── 1/1355 2.2it/s 0.1s<10:24

...

      48/50     0.688G     0.3705     0.2425     0.8676         11        640: 100% ━━━━━━━━━━━━ 1355/1355 15.6it/s 1:270.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 16.3it/s 13.8s0.1s
                   all       1799       3479      0.972      0.966      0.984      0.901

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      49/50     0.688G     0.3666     0.2509     0.8177          8        640: 0% ──────────── 1/1355 2.1it/s 0.1s<10:30

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

      49/50     0.688G     0.3658     0.2372     0.8644          6        640: 100% ━━━━━━━━━━━━ 1355/1355 15.7it/s 1:270.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 16.1it/s 14.0s0.1s
                   all       1799       3479      0.969       0.97      0.982        0.9

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      50/50     0.688G     0.2605     0.1551     0.8828          6        640: 0% ──────────── 1/1355 2.4it/s 0.1s<9:28

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

      50/50     0.688G     0.3647     0.2344     0.8634          9        640: 100% ━━━━━━━━━━━━ 1355/1355 15.9it/s 1:250.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 16.6it/s 13.5s0.1s
                   all       1799       3479      0.969       0.97      0.982      0.901

50 epochs completed in 1.402 hours.
Optimizer stripped from D:\Inspectra\experiments\bottle\baseline\baseline\weights\last.pt, 6.3MB
Optimizer stripped from D:\Inspectra\experiments\bottle\baseline\baseline\weights\best.pt, 6.3MB

Validating D:\Inspectra\experiments\bottle\baseline\baseline\weights\best.pt...
Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 17.0it/s 13.3s0.1s
             

In [8]:
pcb_result = train_detection_baseline(
    "pcb",
    DETECTION_DATASETS["pcb"]
)

Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\Inspectra\datasets\processed\pcb\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

AMP: checks passed 
WARNING train: Slow image access detected (ping: 0.30.1 ms, read: 3.71.6 MB/s, size: 94.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
train: Scanning D:\Inspectra\datasets\processed\pcb\train\labels... 6370 images, 2164 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8534/8534 303.8it/s 28.1s0.1ss
train: New cache created: D:\Inspectra\datasets\processed\pcb\train\labels.cache
WARNING val: Slow image access detected (ping: 0.20.1 ms, read: 10.12.0 MB/s, size: 105.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\Inspectra\datasets\processed\pcb\val\labels... 802 images, 264 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1066/1066 277.5it/s 3.8s0.1s
val: New cache created: D:\Inspectra\datasets\processed\pcb\val\labels.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

       1/50     0.633G      2.323      4.967      1.402          4        640: 100% ━━━━━━━━━━━━ 2134/2134 15.5it/s 2:180.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 17.1it/s 7.8s0.1s
                   all       1066       1595      0.571      0.616      0.578      0.252

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50     0.729G      2.114      3.076      1.383          9        640: 0% ──────────── 1/2134 2.2it/s 0.1s<16:00

...

      48/50      0.73G      1.488     0.7041      1.065          4        640: 100% ━━━━━━━━━━━━ 2134/2134 16.2it/s 2:120.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 18.0it/s 7.5s0.1s
                   all       1066       1595      0.973      0.979      0.985      0.548

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      49/50      0.73G      1.388     0.5552      1.008          5        640: 0% ──────────── 1/2134 2.4it/s 0.1s<14:54

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

      49/50      0.73G      1.472     0.7074      1.062          1        640: 100% ━━━━━━━━━━━━ 2134/2134 16.2it/s 2:120.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 18.2it/s 7.4s0.1s
                   all       1066       1595      0.978      0.976      0.987       0.55

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      50/50      0.73G      1.588     0.5889       1.24          6        640: 0% ──────────── 1/2134 2.4it/s 0.1s<14:49

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

      50/50      0.73G      1.462     0.7805      1.058          3        640: 100% ━━━━━━━━━━━━ 2134/2134 16.2it/s 2:120.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 18.1it/s 7.4s0.1s
                   all       1066       1595      0.978      0.974      0.987      0.551

50 epochs completed in 1.959 hours.
Optimizer stripped from D:\Inspectra\experiments\pcb\baseline\baseline\weights\last.pt, 6.3MB
Optimizer stripped from D:\Inspectra\experiments\pcb\baseline\baseline\weights\best.pt, 6.3MB

Validating D:\Inspectra\experiments\pcb\baseline\baseline\weights\best.pt...
Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 18.7it/s 7.2s0.1s
                   all  

In [9]:
detection_results = {}

for dataset, config in DETECTION_DATASETS.items():

    if dataset in [
        "bottle",
        "pcb",
    ]:
        continue

    if not config["classes"]:
        print(
            f"\n{dataset}: "
            "classes not prepared yet"
        )
        continue

    print(
        f"\n{'=' * 70}"
    )

    print(
        f"BASELINE: {dataset.upper()}"
    )

    print(
        f"{'=' * 70}"
    )

    result = train_detection_baseline(
        dataset,
        config
    )

    detection_results[
        dataset
    ] = result


textile: classes not prepared yet

BASELINE: WELDING
welding: no training images


In [10]:
baseline_summary = []

for dataset in DETECTION_DATASETS:

    result = readiness_df[
        readiness_df["dataset"]
        == dataset
    ]

    if result.empty:
        continue

    row = result.iloc[0]

    baseline_summary.append(
        {
            "dataset": dataset,
            "task": "detection",
            "ready": bool(
                row["ready"]
            ),
            "train_images": int(
                row["train"]
            ),
            "val_images": int(
                row["val"]
            ),
            "test_images": int(
                row["test"]
            ),
        }
    )


baseline_summary_df = pd.DataFrame(
    baseline_summary
)

display(
    baseline_summary_df
)

baseline_summary_df.to_csv(
    EXPERIMENTS_DIR
    / "baseline_readiness.csv",
    index=False
)

,dataset,task,ready,train_images,val_images,test_images
0,bottle,detection,True,5420,1799,920
1,pcb,detection,True,8534,1066,1068
2,textile,detection,False,0,0,0
3,welding,detection,False,0,0,0


In [11]:
import torchvision
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

In [12]:
ROAD_IMAGE_SIZE = 224

road_train_transform = transforms.Compose([
    transforms.Resize(
        (ROAD_IMAGE_SIZE, ROAD_IMAGE_SIZE)
    ),
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])

road_eval_transform = transforms.Compose([
    transforms.Resize(
        (ROAD_IMAGE_SIZE, ROAD_IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])

In [13]:
road_root = (
    PROCESSED_DATA
    / "road"
)

road_train = datasets.ImageFolder(
    road_root / "train",
    transform=road_train_transform
)

road_val = datasets.ImageFolder(
    road_root / "val",
    transform=road_eval_transform
)

road_test = datasets.ImageFolder(
    road_root / "test",
    transform=road_eval_transform
)

print(
    "Classes:",
    road_train.classes
)

print(
    "Train:",
    len(road_train)
)

print(
    "Val:",
    len(road_val)
)

print(
    "Test:",
    len(road_test)
)

Classes: ['Negative', 'Positive']
Train: 28000
Val: 6000
Test: 6000


In [14]:
ROAD_BATCH_SIZE = 64

road_train_loader = DataLoader(
    road_train,
    batch_size=ROAD_BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

road_val_loader = DataLoader(
    road_val,
    batch_size=ROAD_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

road_test_loader = DataLoader(
    road_test,
    batch_size=ROAD_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

In [15]:
from torchvision.models import (
    resnet18,
    ResNet18_Weights,
)

road_model = resnet18(
    weights=ResNet18_Weights.DEFAULT
)

road_model.fc = torch.nn.Linear(
    road_model.fc.in_features,
    len(road_train.classes)
)

road_model = road_model.to(
    DEVICE
)

print(
    road_model
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Garvit/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:07<00:00, 6.24MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [16]:
road_criterion = torch.nn.CrossEntropyLoss()

road_optimizer = torch.optim.AdamW(
    road_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

In [17]:
def train_classification_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(
            device
        )

        labels = labels.to(
            device
        )

        optimizer.zero_grad()

        outputs = model(
            images
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = (
            outputs.argmax(
                dim=1
            )
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    return (
        total_loss / total,
        correct / total
    )

In [18]:
@torch.no_grad()
def evaluate_classification(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(
            device
        )

        labels = labels.to(
            device
        )

        outputs = model(
            images
        )

        loss = criterion(
            outputs,
            labels
        )

        total_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = (
            outputs.argmax(
                dim=1
            )
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    return (
        total_loss / total,
        correct / total
    )

In [19]:
ROAD_EPOCHS = 10

road_history = []

best_val_accuracy = 0

road_model_dir = (
    MODELS_DIR
    / "road"
)

road_model_dir.mkdir(
    parents=True,
    exist_ok=True
)

for epoch in range(
    ROAD_EPOCHS
):

    train_loss, train_accuracy = (
        train_classification_epoch(
            road_model,
            road_train_loader,
            road_criterion,
            road_optimizer,
            DEVICE
        )
    )

    val_loss, val_accuracy = (
        evaluate_classification(
            road_model,
            road_val_loader,
            road_criterion,
            DEVICE
        )
    )

    road_history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        }
    )

    print(
        f"Epoch {epoch + 1:02d}/{ROAD_EPOCHS} "
        f"| "
        f"train_loss={train_loss:.4f} "
        f"| "
        f"train_acc={train_accuracy:.4f} "
        f"| "
        f"val_loss={val_loss:.4f} "
        f"| "
        f"val_acc={val_accuracy:.4f}"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = (
            val_accuracy
        )

        torch.save(
            road_model.state_dict(),
            road_model_dir
            / "baseline_resnet18.pt"
        )

Epoch 01/10 | train_loss=0.0105 | train_acc=0.9969 | val_loss=0.0024 | val_acc=0.9995
Epoch 02/10 | train_loss=0.0038 | train_acc=0.9988 | val_loss=0.0034 | val_acc=0.9993
Epoch 03/10 | train_loss=0.0017 | train_acc=0.9996 | val_loss=0.0036 | val_acc=0.9993
Epoch 04/10 | train_loss=0.0024 | train_acc=0.9992 | val_loss=0.0042 | val_acc=0.9988
Epoch 05/10 | train_loss=0.0024 | train_acc=0.9993 | val_loss=0.0039 | val_acc=0.9987
Epoch 06/10 | train_loss=0.0021 | train_acc=0.9992 | val_loss=0.0025 | val_acc=0.9995
Epoch 07/10 | train_loss=0.0010 | train_acc=0.9995 | val_loss=0.0027 | val_acc=0.9995
Epoch 08/10 | train_loss=0.0027 | train_acc=0.9992 | val_loss=0.0019 | val_acc=0.9995
Epoch 09/10 | train_loss=0.0007 | train_acc=0.9999 | val_loss=0.0035 | val_acc=0.9990
Epoch 10/10 | train_loss=0.0012 | train_acc=0.9996 | val_loss=0.0026 | val_acc=0.9992


In [20]:
road_history_df = pd.DataFrame(
    road_history
)

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    road_history_df["epoch"],
    road_history_df["train_accuracy"],
    label="Train Accuracy"
)

plt.plot(
    road_history_df["epoch"],
    road_history_df["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Road Classification Baseline"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()

<Figure size 1000x500 with 1 Axes>